# MedIntel AI: Synthetic Medication & Supply-Chain Dataset Exploration

This notebook demonstrates loading, inspecting, and validating the synthetic medication supply-chain datasets generated for the **MedIntel AI** clinical intelligence platform.

In [ ]:
import os
from pathlib import Path
import pandas as pd
import numpy as np

data_dir = Path("../data/raw")
print("Available CSV files:")
for f in sorted(data_dir.glob("*.csv")):
    df_tmp = pd.read_csv(f)
    print(f"  • {f.name:25s}: {len(df_tmp):>8,} rows, {len(df_tmp.columns):>2} cols")

## 1. Inspect Master Catalogs (Medications & Locations)

In [ ]:
meds_df = pd.read_csv(data_dir / "medications.csv")
locs_df = pd.read_csv(data_dir / "locations.csv")
sups_df = pd.read_csv(data_dir / "suppliers.csv")

print("=== Medications Sample (Total:", len(meds_df), ") ===")
display(meds_df.head(10))

print("\n=== Healthcare Facilities Sample (Total:", len(locs_df), ") ===")
display(locs_df)

## 2. Daily Utilization & Inventory Dynamics

In [ ]:
util_df = pd.read_csv(data_dir / "utilization_daily.csv")
inv_df = pd.read_csv(data_dir / "inventory.csv")
po_df = pd.read_csv(data_dir / "purchase_orders.csv")
events_df = pd.read_csv(data_dir / "supplier_events.csv")
lots_df = pd.read_csv(data_dir / "expiry_lots.csv")
gt_df = pd.read_csv(data_dir / "ground_truth.csv")

print(f"Daily utilization records: {len(util_df):,}")
print(f"Daily inventory snapshots: {len(inv_df):,}")
print(f"Purchase orders count:     {len(po_df):,}")
print(f"Supplier events recorded:  {len(events_df):,}")
print(f"Active lot records:        {len(lots_df):,}")

## 3. Benchmark Ground Truth Evaluation Scenarios

In [ ]:
print("=== Ground Truth Scenarios Benchmark ===")
display(gt_df)

## 4. Scenario 1 Deep Dive: Norepinephrine Surge & Carrier Delay

Investigating Norepinephrine (`MED001`) at Valley Regional Trauma Center (`LOC006`).

In [ ]:
s1_util = util_df[(util_df["medication_id"] == "MED001") & (util_df["location_id"] == "LOC006")].sort_values("date")
s1_inv = inv_df[(inv_df["medication_id"] == "MED001") & (inv_df["location_id"] == "LOC006")].sort_values("snapshot_date")
s1_pos = po_df[(po_df["medication_id"] == "MED001") & (po_df["location_id"] == "LOC006")]
s1_evts = events_df[(events_df["medication_id"] == "MED001")]

print("Recent utilization trend (last 10 days):")
display(s1_util.tail(10))

print("\nFinal snapshot inventory status:")
display(s1_inv.tail(5))

print("\nPurchase orders:")
display(s1_pos.tail(5))

print("\nActive supplier disruption events:")
display(s1_evts)

## 5. Scenario 3 Deep Dive: Inter-Facility Inventory Imbalance

Comparing Dexmedetomidine (`MED008`) inventory between Westside Community (`LOC005`) and Central AMC (`LOC001`).

In [ ]:
final_date = "2026-08-31"
dex_loc5 = inv_df[(inv_df["medication_id"] == "MED008") & (inv_df["location_id"] == "LOC005") & (inv_df["snapshot_date"] == final_date)]
dex_loc1 = inv_df[(inv_df["medication_id"] == "MED008") & (inv_df["location_id"] == "LOC001") & (inv_df["snapshot_date"] == final_date)]

print("LOC005 (Deficit Node):")
display(dex_loc5[["inventory_id", "quantity_on_hand", "average_daily_usage", "days_of_supply"]])

print("\nLOC001 (Surplus Node):")
display(dex_loc1[["inventory_id", "quantity_on_hand", "average_daily_usage", "days_of_supply"]])

## 6. Summary

The dataset provides complete, coherent multi-echelon supply-chain data suitable for training and evaluating clinical AI reasoning agents.